# SeineCrops — Sprint S4 : Divergence et phénologie

Deux volets complémentaires exploitant les séries temporelles
Sentinel-2 agrégées en S2 et le classifieur entraîné en S3.

**Divergence** : pour chaque parcelle, calcul d'un score de distance
au profil médian de sa classe déclarée (RPG). Les parcelles au-delà
d'un seuil sont signalées comme divergentes — erreur de déclaration,
culture intermédiaire non déclarée, ou stress exceptionnel.

**Phénologie** : ajustement d'une courbe lissée (Savitzky-Golay) sur
le profil NDVI temporel de chaque parcelle et extraction des métriques
SOS (Start of Season), POS (Peak of Season), EOS (End of Season) et
longueur de saison. Résultats agrégés par zone et par culture,
comparables aux produits HR-VPP Copernicus.

---

## Structure du notebook

| Section | Contenu |
|---------|---------|
| 5.1 | Chargement et pivot — features wide depuis `derived.s2_parcelles_monthly`, labels depuis `derived.rpg_parcelles_aoi` |
| 5.2 | Profils médians par classe et scores de divergence (distance euclidienne, option DTW) |
| 5.3 | Lissage NDVI et extraction des métriques phénologiques (SOS, POS, EOS, LOS) |
| 5.4 | Chargement PostGIS et visualisation — tables `derived.divergence`, `derived.phenologie` |

---

## Dépendances

- `derived.s2_parcelles_monthly` — 13,7 M lignes (sprint S2, `03_series_s2.ipynb`)
- `derived.rpg_parcelles_aoi` — 80 689 parcelles avec `code_cultu` / `code_group` (sprint S1)

> ⚠️ Les données actuelles sont pré-correction SCL. La mécanique est
> validée sur ce jeu bruité ; les seuils et paramètres seront recalibrés
> après le re-run complet de nb03.

In [ ]:
# ── Imports ───────────────────────────────────────────────────────────
import os
from pathlib import Path

import numpy as np
import pandas as pd
import psycopg2
import matplotlib.pyplot as plt
from scipy.signal import savgol_filter
from dotenv import load_dotenv

# ── Racine du projet ──────────────────────────────────────────────────
def find_project_root(marker: str = ".projectroot") -> Path:
    here = Path().resolve()
    for parent in [here, *here.parents]:
        if (parent / marker).exists() or (parent / ".git").exists():
            return parent
    raise FileNotFoundError("Racine du projet introuvable")

PROJECT_ROOT = find_project_root()
load_dotenv(PROJECT_ROOT / ".env")

# ── Connexion PostGIS ─────────────────────────────────────────────────
PG_PARAMS = {
    "host": os.getenv("PG_HOST", "localhost"),
    "port": int(os.getenv("PG_PORT", 5432)),
    "dbname": os.getenv("PG_DBNAME", "seinecrops"),
    "user": os.getenv("PG_USER", "postgres"),
    "password": os.getenv("PG_PASSWORD"),
}

# ── Constantes ────────────────────────────────────────────────────────
MONTHS = [f"{y}-{m:02d}" for y in (2023, 2024)
          for m in range(1, 13)
          if (y == 2023 and m >= 9) or (y == 2024)]  # 16 mois

# Mapping code_group → classe cible (v3 : 8 classes)
# Synchronisé avec nb04 — betterave traitée via code_cultu == "BTN"
GROUP_MAP = {
    1:  "cereales_hiver",      # Blé tendre
    3:  "cereales_hiver",      # Orge
    2:  "mais",                # Maïs grain et ensilage
    5:  "colza",               # Colza
    9:  "lin",                 # Plantes à fibres (≈ lin fibre en Normandie)
    18: "prairie",             # Prairies permanentes
    19: "prairie",             # Prairies temporaires
    25: "legumes_fleurs",      # Légumes ou fleurs
}

### 5.1 — Chargement et pivot

Chargement des features depuis `derived.s2_parcelles_monthly` (format
long) et des labels depuis `derived.rpg_parcelles_aoi`. Pivot en format
wide (une ligne par parcelle × 704 colonnes) et regroupement des cultures
via `GROUP_MAP`. Les parcelles hors dictionnaire sont classées `"autres"`.

In [ ]:
# ── Chargement depuis PostGIS ─────────────────────────────────────────
conn = psycopg2.connect(**PG_PARAMS)

df_long = pd.read_sql("""
    SELECT id_parcel, mois, variable, mean, std, p10, p90
    FROM derived.s2_parcelles_monthly
    ORDER BY id_parcel, mois, variable
""", conn)

df_labels = pd.read_sql("""
    SELECT id_parcel, code_group, code_cultu
    FROM derived.rpg_parcelles_aoi
""", conn)

conn.close()

print(f"Features long : {len(df_long):>12,} lignes")
print(f"Parcelles RPG : {len(df_labels):>12,} parcelles")
print(f"Parcelles S2  : {df_long['id_parcel'].nunique():>7,}")

In [ ]:
# ── Pivot long → wide ─────────────────────────────────────────────────
STATS = ["mean", "std", "p10", "p90"]

df_melt = df_long.melt(
    id_vars=["id_parcel", "mois", "variable"],
    value_vars=STATS,
    var_name="stat",
    value_name="value",
)
df_melt["col"] = df_melt["variable"] + "_" + df_melt["stat"] + "_" + df_melt["mois"]

df_wide = df_melt.pivot_table(
    index="id_parcel", columns="col", values="value"
).reset_index()

# ── Labels : regroupement des cultures ────────────────────────────────
df_labels["code_group"] = pd.to_numeric(df_labels["code_group"], errors="coerce")
df_labels["classe"] = df_labels["code_group"].map(GROUP_MAP).fillna("autres")
# Betterave : code_group 24 partagé avec d'autres cultures industrielles
df_labels.loc[df_labels["code_cultu"] == "BTN", "classe"] = "betterave"

df = df_wide.merge(df_labels[["id_parcel", "classe"]], on="id_parcel", how="inner")

print(f"Matrice wide  : {df.shape[0]:,} parcelles × {df.shape[1] - 2} features")
print(f"\nDistribution des classes :")
print(df["classe"].value_counts().to_string())

### 5.2 — Profils médians et scores de divergence

Pour chaque classe déclarée (RPG), calcul du profil médian sur les
704 features. La **distance euclidienne** de chaque parcelle à son
profil de classe mesure l'écart entre le couvert observé par satellite
et la culture déclarée. Un score élevé signale une divergence :
erreur de déclaration, culture intermédiaire non déclarée, ou stress
agronomique exceptionnel.

Les parcelles au-delà du **95e percentile** intra-classe sont
signalées comme divergentes. Ce seuil est provisoire et sera recalibré
après le re-run sur données dé-bruitées.

In [ ]:
# ── Profil médian par classe déclarée ─────────────────────────────────
feature_cols = [c for c in df.columns if c not in ("id_parcel", "classe")]

medians = df.groupby("classe")[feature_cols].median()

print(f"Profils médians : {medians.shape[0]} classes × {medians.shape[1]} features")
print(f"NaN dans les profils médians : {medians.isna().sum().sum()}")

In [ ]:
# ── Distance euclidienne au profil médian de la classe déclarée ───────
X = df[feature_cols].to_numpy(dtype=np.float64, na_value=np.nan)
classes = df["classe"].values

# Matrice des profils médians alignée sur chaque parcelle
X_med = medians.loc[classes].to_numpy()

# Distance euclidienne — NaN traités comme écart nul (conservatif)
diff = np.where(np.isnan(X) | np.isnan(X_med), 0.0, X - X_med)
df["dist_classe"] = np.sqrt((diff ** 2).sum(axis=1))

print(df["dist_classe"].describe().to_string())

In [ ]:
# ── Seuil de divergence : médiane + 2 × IQR (intra-classe) ───────────
def divergence_stats(g):
    q1, q2, q3 = g.quantile([0.25, 0.50, 0.75])
    seuil = q2 + 2 * (q3 - q1)
    n_div = (g > seuil).sum()
    return pd.Series({"median": q2, "IQR": q3 - q1, "seuil": seuil,
                       "n_div": n_div, "taux": n_div / len(g)})

stats_div = df.groupby("classe")["dist_classe"].apply(divergence_stats).unstack()

# ── Visualisation ─────────────────────────────────────────────────────
classes = stats_div.sort_values("taux", ascending=False).index
n_classes = len(classes)
fig, axes = plt.subplots(2, 4, figsize=(16, 8), sharey=False)
axes = axes.ravel()

for i, cls in enumerate(classes):
    ax = axes[i]
    vals = df.loc[df["classe"] == cls, "dist_classe"]
    seuil = stats_div.loc[cls, "seuil"]
    taux = stats_div.loc[cls, "taux"]

    ax.hist(vals, bins=60, color="steelblue", edgecolor="none", alpha=0.8)
    ax.axvline(seuil, color="firebrick", ls="--", lw=1.5,
               label=f"med + 2·IQR = {seuil:,.0f}")
    ax.set_title(f"{cls}\n{taux:.1%} divergentes", fontsize=10)
    ax.legend(fontsize=7, loc="upper right")
    ax.set_xlabel("distance euclidienne")

# Masquer l'axe restant si < 8 classes
for j in range(n_classes, len(axes)):
    axes[j].set_visible(False)

fig.suptitle("Distribution des distances au profil médian de classe (RPG déclaré)",
             fontsize=12, y=1.01)
plt.tight_layout()
plt.show()

# ── Appliquer le seuil ────────────────────────────────────────────────
df["seuil_div"] = df["classe"].map(stats_div["seuil"])
df["divergent"] = df["dist_classe"] > df["seuil_div"]

n_div = df["divergent"].sum()
print(f"\nParcelles divergentes : {n_div:,} / {len(df):,} "
      f"({100 * n_div / len(df):.1f} %)")
print(f"\nDétail par classe :")
print(stats_div[["median", "IQR", "seuil", "n_div", "taux"]]
        .sort_values("taux", ascending=False)
        .to_string(formatters={"taux": "{:.1%}".format,
                                "median": "{:,.0f}".format,
                                "IQR": "{:,.0f}".format,
                                "seuil": "{:,.0f}".format,
                                "n_div": "{:,.0f}".format}))

In [ ]:
# ── Les deux populations sont-elles géographiquement séparées ? ───────
# Seuil approximatif entre les deux modes
seuil_bimodal = 30000
df["pop"] = np.where(df["dist_classe"] < seuil_bimodal, "proche", "loin")

# Centroïde Y (latitude Lambert-93) par parcelle
conn = psycopg2.connect(**PG_PARAMS)
df_geo = pd.read_sql("""
    SELECT id_parcel, ST_Y(ST_Centroid(geom)) AS y_centroid
    FROM derived.rpg_parcelles_aoi
""", conn)
conn.close()

df_diag = df[["id_parcel", "pop", "dist_classe"]].merge(df_geo, on="id_parcel")

fig, ax = plt.subplots(figsize=(8, 5))
for pop, color in [("proche", "steelblue"), ("loin", "firebrick")]:
    sub = df_diag[df_diag["pop"] == pop]
    ax.scatter(sub["y_centroid"], sub["dist_classe"], s=1, alpha=0.3,
               color=color, label=pop)
ax.set_xlabel("Y centroïde (Lambert-93, m)")
ax.set_ylabel("Distance au profil médian")
ax.legend()
ax.set_title("Distance vs latitude — split nord/sud ?")
plt.tight_layout()
plt.show()

print(df_diag.groupby("pop")["y_centroid"].describe().to_string())

In [ ]:
# ── Split est/ouest ? ─────────────────────────────────────────────────
conn = psycopg2.connect(**PG_PARAMS)
df_geo = pd.read_sql("""
    SELECT id_parcel,
           ST_X(ST_Centroid(geom)) AS x_centroid,
           ST_Y(ST_Centroid(geom)) AS y_centroid
    FROM derived.rpg_parcelles_aoi
""", conn)
conn.close()

df_diag = df[["id_parcel", "pop", "dist_classe"]].merge(df_geo, on="id_parcel")

fig, ax = plt.subplots(figsize=(8, 5))
for pop, color in [("proche", "steelblue"), ("loin", "firebrick")]:
    sub = df_diag[df_diag["pop"] == pop]
    ax.scatter(sub["x_centroid"], sub["dist_classe"], s=1, alpha=0.3,
               color=color, label=pop)
ax.set_xlabel("X centroïde (Lambert-93, m)")
ax.set_ylabel("Distance au profil médian")
ax.legend()
ax.set_title("Distance vs longitude — split est/ouest ?")
plt.tight_layout()
plt.show()

print(df_diag.groupby("pop")["x_centroid"].describe().to_string())

In [ ]:
# ── Carte XY colorée par population ───────────────────────────────────
fig, ax = plt.subplots(figsize=(10, 8))
for pop, color in [("proche", "steelblue"), ("loin", "firebrick")]:
    sub = df_diag[df_diag["pop"] == pop]
    ax.scatter(sub["x_centroid"], sub["y_centroid"], s=0.5, alpha=0.3,
               color=color, label=pop)
ax.set_xlabel("X (Lambert-93)")
ax.set_ylabel("Y (Lambert-93)")
ax.legend(markerscale=10)
ax.set_title("Répartition spatiale des deux populations de distance")
ax.set_aspect("equal")
plt.tight_layout()
plt.show()

print(f"Proches : {(df_diag['pop']=='proche').sum():,}")
print(f"Loin    : {(df_diag['pop']=='loin').sum():,}")

In [ ]:
# ── Standardisation + recalcul de distance ────────────────────────────
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()
X_scaled = scaler.fit_transform(df[feature_cols].to_numpy(dtype=np.float64))

# Profils médians standardisés par classe
df_scaled = pd.DataFrame(X_scaled, columns=feature_cols)
df_scaled["classe"] = df["classe"].values
medians_scaled = df_scaled.groupby("classe")[feature_cols].median()

# Distance euclidienne standardisée
X_med_scaled = medians_scaled.loc[df["classe"].values].to_numpy()
df["dist_classe"] = np.sqrt(((X_scaled - X_med_scaled) ** 2).sum(axis=1))

# ── Même visu que précédemment ────────────────────────────────────────
def divergence_stats(g):
    q1, q2, q3 = g.quantile([0.25, 0.50, 0.75])
    seuil = q2 + 2 * (q3 - q1)
    n_div = (g > seuil).sum()
    return pd.Series({"median": q2, "IQR": q3 - q1, "seuil": seuil,
                       "n_div": n_div, "taux": n_div / len(g)})

stats_div = df.groupby("classe")["dist_classe"].apply(divergence_stats).unstack()

classes = stats_div.sort_values("taux", ascending=False).index
fig, axes = plt.subplots(2, 4, figsize=(16, 8), sharey=False)
axes = axes.ravel()

for i, cls in enumerate(classes):
    ax = axes[i]
    vals = df.loc[df["classe"] == cls, "dist_classe"]
    seuil = stats_div.loc[cls, "seuil"]
    taux = stats_div.loc[cls, "taux"]
    ax.hist(vals, bins=60, color="steelblue", edgecolor="none", alpha=0.8)
    ax.axvline(seuil, color="firebrick", ls="--", lw=1.5,
               label=f"med + 2·IQR = {seuil:.1f}")
    ax.set_title(f"{cls}\n{taux:.1%} divergentes", fontsize=10)
    ax.legend(fontsize=7, loc="upper right")
    ax.set_xlabel("distance euclidienne (standardisée)")

for j in range(len(classes), len(axes)):
    axes[j].set_visible(False)

fig.suptitle("Distance au profil médian — features standardisées",
             fontsize=12, y=1.01)
plt.tight_layout()
plt.show()

print(stats_div[["median", "IQR", "seuil", "n_div", "taux"]]
        .sort_values("taux", ascending=False)
        .to_string(formatters={"taux": "{:.1%}".format}))